In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display
from sklearn.metrics import auc, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score, roc_curve


In [ ]:
VALIDATION_LOOP_DIR = Path(".").resolve()
REPO_DIR = VALIDATION_LOOP_DIR.parent

sys.path.insert(0, str((REPO_DIR / "training_loop").resolve()))
sys.path.insert(0, str(REPO_DIR.resolve()))

OUTPUT_DIR = REPO_DIR / "output" / "history_3d_simple_cnn"
TRAIN_PREDICT_CSV = OUTPUT_DIR / "predict_train_3d_simple_cnn.csv"
VAL_PREDICT_CSV = OUTPUT_DIR / "predict_val_3d_simple_cnn.csv"


In [ ]:
train_predictions = pd.read_csv(TRAIN_PREDICT_CSV)
val_predictions = pd.read_csv(VAL_PREDICT_CSV)

train_predictions.head(), val_predictions.head()


In [ ]:
def compute_auc_metrics(predictions_df):
    y_true = predictions_df["target"].to_numpy()
    y_score = predictions_df["predict"].to_numpy()

    precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_score)

    return {
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc": auc(recall_curve, precision_curve),
    }


auc_metrics_df = pd.DataFrame(
    [
        compute_auc_metrics(train_predictions),
        compute_auc_metrics(val_predictions),
    ],
    index=["Train", "Validation"],
)
auc_metrics_df


In [ ]:
def plot_preliminary_plots(y_true, y_score, dataset_name):
    fig_hist = px.histogram(
        x=y_score,
        color=y_true.astype(str),
        nbins=50,
        labels={"color": "True Labels", "x": "Score"},
        title=f"{dataset_name}: Histogram of Scores",
        width=800,
        height=500,
    )
    fig_hist.show()

    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    df_thresh = pd.DataFrame({"FPR": fpr, "TPR": tpr}, index=thresholds)

    fig_thresh = px.line(
        df_thresh,
        title=f"{dataset_name}: TPR and FPR at every threshold",
        width=800,
        height=500,
    )
    fig_thresh.update_yaxes(scaleanchor="x", scaleratio=1)
    fig_thresh.update_xaxes(range=[0, 1], constrain="domain")
    fig_thresh.show()


def plot_roc_curve(y_true, y_score, dataset_name):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)

    fig = px.area(
        x=fpr,
        y=tpr,
        title=f"{dataset_name}: ROC Curve (AUC={roc_auc:.3f})",
        labels={"x": "False Positive Rate", "y": "True Positive Rate"},
        width=800,
        height=500,
    )
    fig.add_shape(type="line", line=dict(dash="dash"), x0=0, x1=1, y0=0, y1=1)
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.show()


def plot_pr_curve(y_true, y_score, dataset_name):
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    pr_auc = auc(recall, precision)

    fig = px.area(
        x=recall,
        y=precision,
        title=f"{dataset_name}: Precision-Recall Curve (AUC={pr_auc:.3f})",
        labels={"x": "Recall", "y": "Precision"},
        width=800,
        height=500,
    )
    fig.add_shape(type="line", line=dict(dash="dash"), x0=0, x1=1, y0=1, y1=0)
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.show()


In [ ]:
for predictions_df, name in [
    (train_predictions, "Train"),
    (val_predictions, "Validation"),
]:
    y_true = predictions_df["target"].to_numpy()
    y_score = predictions_df["predict"].to_numpy()

    print(name)
    display(pd.DataFrame([compute_auc_metrics(predictions_df)]))
    plot_preliminary_plots(y_true, y_score, name)
    plot_roc_curve(y_true, y_score, name)
    plot_pr_curve(y_true, y_score, name)


In [ ]:
threshold = 0.5


In [ ]:
def compute_threshold_metrics(predictions_df, threshold):
    y_true = predictions_df["target"].to_numpy()
    y_score = predictions_df["predict"].to_numpy()
    y_pred = (y_score >= threshold).astype(int)

    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


threshold_metrics_df = pd.DataFrame(
    [
        compute_threshold_metrics(train_predictions, threshold),
        compute_threshold_metrics(val_predictions, threshold),
    ],
    index=["Train", "Validation"],
)
threshold_metrics_df
